<a href="https://colab.research.google.com/github/dssg/MLinPractice/blob/main/Adversarial_Validation_Audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title Common imports & settings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, accuracy_score, brier_score_loss,
    classification_report, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

import matplotlib.pyplot as plt

# %matplotlib inline  # If needed

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
# @title Activity 1 – Adversarial Validation Audit Skeleton

# 1. Load / simulate train vs live data
# Replace with actual paths or data loading as needed.
# For example: upload via Colab then read with pd.read_csv(...)
# from google.colab import files
# uploaded = files.upload()

# TODO: replace with real paths or upload logic
X_train = pd.read_csv("X_train.csv")  # shape (n_train, d)
X_live = pd.read_csv("X_live.csv")    # shape (n_live, d)

print("Train shape:", X_train.shape)
print("Live shape:", X_live.shape)

# 2. Create adversarial dataset: label train=0, live=1
X_adv = pd.concat([X_train, X_live], axis=0, ignore_index=True)
y_adv = np.concatenate([
    np.zeros(len(X_train), dtype=int),
    np.ones(len(X_live), dtype=int)
])

print("Adversarial dataset shape:", X_adv.shape, " Labels:", y_adv.shape)

# 3. Train/val split for adversarial classifier
X_tr, X_te, y_tr, y_te = train_test_split(
    X_adv, y_adv, test_size=0.2, random_state=RANDOM_STATE, stratify=y_adv
)

# 4. Define adversarial classifier (students can try RF, XGBoost, etc.)
adv_clf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

# TODO: students can tune hyperparameters
adv_clf.fit(X_tr, y_tr)

y_pred_proba = adv_clf.predict_proba(X_te)[:, 1]
adv_auc = roc_auc_score(y_te, y_pred_proba)
print(f"Adversarial AUC (train vs live): {adv_auc:.3f}")

# 5. Feature importance
feature_importances = pd.Series(
    adv_clf.feature_importances_,
    index=X_adv.columns
).sort_values(ascending=False)

print("Top drift-driving features:")
print(feature_importances.head(20))

# TODO:
# - Is there strong distribution shift? (AUC threshold)
# - Which features drive the shift?